# RNN, LSTM, and GRU — Sequential Deep Learning
Recurrent Neural Networks (RNNs) and their variants are the foundation of sequential and time-series modeling in deep learning. This notebook covers RNNs, LSTMs, GRUs, and Bidirectional architectures.

## 1. Recurrent Neural Networks (RNNs)
An RNN is a neural network designed to work with sequential data by maintaining a **hidden state** that is updated at each time step.

At each time step `t`:
```
h_t = tanh(W_h * h_{t-1} + W_x * x_t + b)
y_t = W_y * h_t
```
- `h_t` is the hidden state at time `t` (memory of past)
- `x_t` is the current input
- The same weights are shared across all time steps

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# Simple RNN for sequence classification
model = models.Sequential([
    layers.Input(shape=(50, 1)),             # 50 time steps, 1 feature
    layers.SimpleRNN(64, return_sequences=False),
    layers.Dense(1, activation='sigmoid')
])
model.summary()

Model: "sequential"

┏━━━━━━━━━┳━━━━━━━┳━━━━┓
┃ Layer   ┃ Outp… ┃ P… ┃
┃ (type)  ┃ Shape ┃  # ┃
┡━━━━━━━━━╇━━━━━━━╇━━━━┩
│ simple… │ (Non… │ 4… │
│ (Simpl… │ 64)   │    │
├─────────┼───────┼────┤
│ dense   │ (Non… │ 65 │
│ (Dense) │ 1)    │    │
└─────────┴───────┴────┘

 Total params: 4,289 (16.75 KB)

 Trainable params: 4,289 (16.75 KB)

 Non-trainable params: 0 (0.00 B)

## 2. The Vanishing Gradient Problem
The fundamental weakness of vanilla RNNs. During backpropagation through time (BPTT), the gradient is multiplied by the weight matrix at each step. With many time steps:
- If weights < 1: gradients **vanish** → early time steps learn nothing
- If weights > 1: gradients **explode** → training diverges

This limits vanilla RNNs to learning short-range dependencies only.

## 3. Long Short-Term Memory (LSTM)
Introduced by Hochreiter & Schmidhuber (1997) to solve the vanishing gradient problem. LSTMs use **gates** and a dedicated **cell state** (`C_t`) to carry information across long sequences.

**The Four Equations of LSTM:**
- **Forget Gate** `f_t = σ(W_f·[h_{t-1}, x_t] + b_f)` — what to erase from cell state
- **Input Gate** `i_t = σ(W_i·[h_{t-1}, x_t] + b_i)` — what new info to store
- **Cell Candidate** `C̃_t = tanh(W_C·[h_{t-1}, x_t] + b_C)` — new cell candidate
- **Cell State Update** `C_t = f_t * C_{t-1} + i_t * C̃_t`
- **Output Gate** `o_t = σ(W_o·[h_{t-1}, x_t] + b_o)` — what to output
- **Hidden State** `h_t = o_t * tanh(C_t)`

In [2]:
# LSTM for Time Series Prediction
lstm_model = models.Sequential([
    layers.Input(shape=(100, 5)),          # 100 time steps, 5 features
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(64, return_sequences=False),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)                         # Regression output
])
lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━┳━━━━━━━┳━━━━┓
┃ Layer   ┃ Outp… ┃ P… ┃
┃ (type)  ┃ Shape ┃  # ┃
┡━━━━━━━━━╇━━━━━━━╇━━━━┩
│ lstm    │ (Non… │ 6… │
│ (LSTM)  │ 100,  │    │
│         │ 128)  │    │
├─────────┼───────┼────┤
│ dropout │ (Non… │  0 │
│ (Dropo… │ 100,  │    │
│         │ 128)  │    │
├─────────┼───────┼────┤
│ lstm_1  │ (Non… │ 4… │
│ (LSTM)  │ 64)   │    │
├─────────┼───────┼────┤
│ dense_1 │ (Non… │ 2… │
│ (Dense) │ 32)   │    │
├─────────┼───────┼────┤
│ dense_2 │ (Non… │ 33 │
│ (Dense) │ 1)    │    │
└─────────┴───────┴────┘

 Total params: 120,129 (469.25 KB)

 Trainable params: 120,129 (469.25 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Gated Recurrent Unit (GRU)
Introduced by Cho et al. (2014) as a simplified alternative to LSTM. GRU merges the forget and input gates into a single **Update Gate** and eliminates the separate cell state.

**GRU Equations:**
- **Reset Gate** `r_t = σ(W_r·[h_{t-1}, x_t])` — how much past to forget
- **Update Gate** `z_t = σ(W_z·[h_{t-1}, x_t])` — how much to update state
- **Candidate** `h̃_t = tanh(W·[r_t * h_{t-1}, x_t])`
- **New State** `h_t = (1 - z_t) * h_{t-1} + z_t * h̃_t`

GRUs train faster with roughly equal performance to LSTMs on most tasks.

In [3]:
# Side-by-side comparison
inputs = layers.Input(shape=(50, 10))

# LSTM branch
lstm_out = layers.LSTM(64)(inputs)
# GRU branch
gru_out  = layers.GRU(64)(inputs)

print("LSTM output shape:", lstm_out.shape)
print("GRU  output shape:", gru_out.shape)
print("\nKey difference: GRU has fewer parameters (2 gates vs 3 gates in LSTM)")

LSTM output shape: (None, 64)
GRU  output shape: (None, 64)

Key difference: GRU has fewer parameters (2 gates vs 3 gates in LSTM)


## 5. Bidirectional RNNs
A standard RNN only sees the past context. A **Bidirectional RNN** wraps any recurrent layer to process the sequence in **both directions** — forward and backward — and concatenates the results.

This is especially powerful for NLP tasks (e.g., Named Entity Recognition) where understanding a word requires both left *and* right context.

```
Forward RNN:  x_1 → x_2 → x_3 → ... → x_T
Backward RNN: x_T → x_{T-1} → ... → x_1
```

In [4]:
# Bidirectional LSTM — common in NLP
bi_model = models.Sequential([
    layers.Input(shape=(30, 128)),            # 30 words, 128-dim embeddings
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(5, activation='softmax')     # 5-class NER tags
])
bi_model.summary()

Model: "sequential_2"

┏━━━━━━━━━┳━━━━━━━┳━━━━┓
┃ Layer   ┃ Outp… ┃ P… ┃
┃ (type)  ┃ Shape ┃  # ┃
┡━━━━━━━━━╇━━━━━━━╇━━━━┩
│ bidire… │ (Non… │ 9… │
│ (Bidir… │ 30,   │    │
│         │ 128)  │    │
├─────────┼───────┼────┤
│ bidire… │ (Non… │ 4… │
│ (Bidir… │ 64)   │    │
├─────────┼───────┼────┤
│ dense_3 │ (Non… │ 3… │
│ (Dense) │ 5)    │    │
└─────────┴───────┴────┘

 Total params: 140,357 (548.27 KB)

 Trainable params: 140,357 (548.27 KB)

 Non-trainable params: 0 (0.00 B)

# Conclusions and Key Takeaways
Sequential modeling has evolved from the simple but gradient-challenged vanilla RNN, to the powerful but heavy LSTM, to the streamlined GRU.
- **Vanilla RNN**: Fast but can't learn long dependencies (vanishing gradients).
- **LSTM**: The cell state `C_t` acts as a conveyor belt, carrying useful information across hundreds of time steps.
- **GRU**: Fewer gates (2 vs 3), fewer parameters, comparable performance — often the pragmatic choice.
- **Bidirectional**: Doubles the context for tasks where the full sequence is available (NLP tagging, translation).

# Pros and Cons

**Pros:**
- Naturally handle variable-length sequences.
- LSTMs can theoretically capture very long-range dependencies.
- Weight sharing across time steps reduces parameters drastically compared to unrolled architectures.
- Bidirectional variants are exceptionally powerful for NLP tasks.

**Cons:**
- Sequential computation cannot be parallelized (each time step depends on the previous one), making training much slower than Transformers.
- Even LSTMs struggle practically with sequences of thousands of steps.
- More complex to debug and tune than feedforward networks.
- Largely superseded by Transformer-based models in NLP benchmarks.

# 15 Interview Questions and Answers

1. **What is the main limitation of a vanilla RNN?**
   *Answer*: The vanishing gradient problem. During backpropagation through time (BPTT), gradients shrink exponentially as they flow back through many time steps, preventing the network from learning long-range dependencies.

2. **How does LSTM solve the vanishing gradient problem?**
   *Answer*: By introducing an explicit **cell state** (`C_t`) that acts as a memory highway running across time steps. The forget and input gates use sigmoid activations to selectively allow information to pass through unchanged, keeping gradients from collapsing.

3. **What is the role of the Forget Gate in an LSTM?**
   *Answer*: It decides what portion of the previous cell state `C_{t-1}` to erase. A value of 1 keeps everything; a value of 0 discards everything.

4. **What is the role of the Input Gate?**
   *Answer*: It decides what new information from the current time step `(x_t, h_{t-1})` to write into the cell state.

5. **What is the Output Gate in LSTM?**
   *Answer*: It decides what part of the filtered cell state to output as the new hidden state `h_t`.

6. **What are the key differences between LSTM and GRU?**
   *Answer*: GRU merges the forget and input gates into a single Update Gate and eliminates the separate cell state, reducing the number of parameters. This makes GRU faster to train with comparable performance on most tasks.

7. **When would you prefer LSTM over GRU?**
   *Answer*: Generally, LSTMs might have a slight edge on tasks requiring very long-range dependencies (e.g., language modeling on very long texts). However, empirical results are mixed and task-specific.

8. **What is Backpropagation Through Time (BPTT)?**
   *Answer*: The algorithm for training RNNs. The RNN is unrolled across T time steps, creating a computational graph T layers deep, and standard backpropagation is applied. This is why gradient vanishing/explosion is especially severe.

9. **What is gradient clipping and when do you use it?**
   *Answer*: Capping the gradient norm if it exceeds a threshold before applying the update. It's a standard technique for combating gradient explosion in RNNs, ensuring parameter updates remain reasonable.

10. **What does `return_sequences=True` mean in Keras?**
    *Answer*: It makes the LSTM/GRU layer return the hidden state at every time step rather than just the last one. Required when stacking multiple recurrent layers or when you need per-step predictions.

11. **What is a Bidirectional RNN and when is it useful?**
    *Answer*: A Bidirectional RNN processes the input sequence both forward and backward and concatenates the two hidden states. It's useful when you have access to the full sequence (not streaming), like in NLP tagging, translation, and speech recognition.

12. **Why are RNNs/LSTMs slower to train than Transformers?**
    *Answer*: Because RNN computation is inherently sequential — each time step must be computed after the previous one is finished. Transformers, using attention, can compute all time step interactions in parallel on a GPU.

13. **How do you handle sequences of different lengths in an RNN?**
    *Answer*: By padding shorter sequences to the same length as the longest sequence and using masking (telling the model to ignore the padded positions when computing the loss).

14. **What is Stateful vs. Stateless mode in Keras LSTMs?**
    *Answer*: In stateless mode (default), the hidden state is reset to zero at the beginning of each batch. In stateful mode, the state is carried over between batches, useful for very long time series that are split into consecutive chunks.

15. **Can an LSTM be used for image captioning?**
    *Answer*: Yes! A CNN encoder extracts a fixed-size feature vector from the image. This vector is used to initialize the hidden state of an LSTM decoder, which then generates the caption word by word using an autoregressive approach.
